# AI 视频生产平台 · Colab ComfyUI 省算力一键启动器（生图 + 4x 动漫超分）

专为 **Google AI Pro（200 Compute Units / 月）** 打造的**单按钮一键启动**工作流：
1. **省钱选卡**：顶部菜单 `代码执行程序 → 更改运行时类型` 选 **L4 GPU**（24GB 显存，约 1.8 单元/小时，够跑 100+ 小时）。
2. **只需点 1 个按钮**：直接点下方 **「1. 一键启动」**（或按 `⌘ + F9`），自动完成挂载 Google Drive → 安装 ComfyUI → 按需下载动漫大模型与超分模型 → 启动 ComfyUI → 生成 HTTPS 隧道链接。
3. **模型永久存 Drive**：模型统一保存在 `MyDrive/AI_Video_Platform_ComfyUI/models`，首次下载后每次开机秒挂载。
4. **闲置自动停表**：默认连续 **15 分钟**无新任务自动释放运行时（`runtime.unassign()`），防止忘关机空烧计算单元。

In [ ]:
#@title 1. 一键启动 ComfyUI（挂载 Drive + 模型准备 + 启动服务 + 出隧道链接 + 闲置自动关机）
DOWNLOAD_ANIMAGINE_XL = True #@param {type:"boolean"}
DOWNLOAD_ANIME_UPSCALE = True #@param {type:"boolean"}
IDLE_SHUTDOWN_MINUTES = 15 #@param {type:"integer"}
EXTRA_MODEL_URL = "" #@param {type:"string"}
EXTRA_MODEL_SUBDIR = "checkpoints" #@param ["checkpoints", "loras", "upscale_models", "vae", "controlnet", "diffusion_models", "text_encoders"]
EXTRA_MODEL_FILENAME = "" #@param {type:"string"}

import os
import re
import json
import time
import shutil
import threading
import subprocess
import urllib.request
from pathlib import Path
from google.colab import drive

DRIVE_ROOT = Path('/content/drive/MyDrive/AI_Video_Platform_ComfyUI')
MODELS_DIR = DRIVE_ROOT / 'models'
COMFY_DIR = Path('/content/ComfyUI')
LOG_PATH = Path('/tmp/comfyui.log')

# --- 第 1 步：挂载 Google Drive 并准备 ComfyUI ---
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

for sub in ['checkpoints', 'loras', 'upscale_models', 'vae', 'controlnet', 'clip', 'clip_vision', 'unet', 'diffusion_models', 'text_encoders']:
    (MODELS_DIR / sub).mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'workflows').mkdir(parents=True, exist_ok=True)

if not COMFY_DIR.exists():
    print('📦 首次初始化 ComfyUI（约 1–2 分钟）...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
    subprocess.run(['pip', 'install', '-q', '-r', str(COMFY_DIR / 'requirements.txt')], check=True)

models_link = COMFY_DIR / 'models'
if models_link.exists() and not models_link.is_symlink():
    shutil.rmtree(models_link)
if not models_link.exists():
    models_link.symlink_to(MODELS_DIR)

if not shutil.which('cloudflared'):
    subprocess.run(['curl', '-sL', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb', '-o', '/tmp/cloudflared.deb'], check=True)
    subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], check=True)

# --- 第 2 步：按需下载模型到 Google Drive（已存在自动跳过） ---
def ensure_file(url: str, dest: Path, min_bytes: int = 1024 * 1024):
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print(f'⏭️  模型已就绪，跳过下载：{dest.name} ({dest.stat().st_size / 1024 / 1024:.1f} MB)')
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f'⬇️  正在下载到 Google Drive：{dest.name}（首次约需 1–3 分钟，后续永久秒加载）...')
    subprocess.run(['curl', '-fL', '-C', '-', url, '-o', str(dest)], check=True)
    print(f'✅ 下载完成：{dest.name} ({dest.stat().st_size / 1024 / 1024:.1f} MB)')

if DOWNLOAD_ANIME_UPSCALE:
    ensure_file(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
        MODELS_DIR / 'upscale_models' / 'RealESRGAN_x4plus_anime_6B.pth',
        min_bytes=10 * 1024 * 1024,
    )

if DOWNLOAD_ANIMAGINE_XL:
    ensure_file(
        'https://huggingface.co/cagliostrolc/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors',
        MODELS_DIR / 'checkpoints' / 'animagine-xl-3.1.safetensors',
        min_bytes=1024 * 1024 * 1024,
    )

if EXTRA_MODEL_URL.strip():
    fname = EXTRA_MODEL_FILENAME.strip() or EXTRA_MODEL_URL.strip().split('?')[0].rstrip('/').split('/')[-1]
    ensure_file(EXTRA_MODEL_URL.strip(), MODELS_DIR / EXTRA_MODEL_SUBDIR / fname, min_bytes=1024)

# --- 第 3 步：启动 ComfyUI + Cloudflare 隧道 + 闲置自动关机守护 ---
subprocess.run(['pkill', '-f', 'ComfyUI/main.py'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
time.sleep(1)

log_file = open(LOG_PATH, 'w')
comfy_proc = subprocess.Popen(
    ['python3', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--dont-print-server'],
    cwd=str(COMFY_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print('🚀 正在启动 ComfyUI，等待 8188 端口就绪...')
ready = False
for _ in range(180):
    if comfy_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    log_file.flush()
    tail = LOG_PATH.read_text(errors='ignore')[-3000:] if LOG_PATH.exists() else ''
    raise RuntimeError(f'ComfyUI 启动失败，日志末尾如下：\n{tail}')
print('✅ ComfyUI 已就绪！正在建立 HTTPS 隧道...')

def idle_watchdog(minutes: int):
    if minutes <= 0:
        return
    last_active = time.time()
    last_history_len = 0
    while True:
        time.sleep(20)
        try:
            with urllib.request.urlopen('http://127.0.0.1:8188/queue', timeout=3) as r:
                q = json.loads(r.read())
            if q.get('queue_running') or q.get('queue_pending'):
                last_active = time.time()
                continue
            with urllib.request.urlopen('http://127.0.0.1:8188/history', timeout=3) as r:
                hist = json.loads(r.read())
            if len(hist) != last_history_len:
                last_history_len = len(hist)
                last_active = time.time()
        except Exception:
            pass
        idle_sec = time.time() - last_active
        if idle_sec >= minutes * 60:
            print(f'\n🛑 已连续闲置 {minutes} 分钟，自动断开并释放 Colab GPU 算力单元！')
            try:
                from google.colab import runtime
                runtime.unassign()
            except Exception:
                os._exit(0)
            return

threading.Thread(target=idle_watchdog, args=(IDLE_SHUTDOWN_MINUTES,), daemon=True).start()

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

url_pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
public_url = None
for line in tunnel.stdout:
    m = url_pattern.search(line)
    if m and not public_url:
        public_url = m.group(0)
        print('\n' + '=' * 64)
        print('🌐 ComfyUI 公网访问地址（填进平台「模型中心 → 引擎实例 → ComfyUI」）：')
        print(f'   {public_url}')
        print(f'⏱️  省算力守护已开启：连续闲置 {IDLE_SHUTDOWN_MINUTES} 分钟后自动释放运行时')
        print('=' * 64 + '\n')


In [ ]:
#@title 2. 用完立刻关机（手动释放 Colab GPU，立刻停止扣除计算单元）
from google.colab import runtime
print('🛑 正在主动释放 Colab GPU 运行时...')
runtime.unassign()
